# PaDiM — Fabric Anomaly Detection (Google Colab)

Implements the pipeline:

`Normal Images -> Pretrained CNN -> Multi-layer Features -> Feature Embedding -> Per-patch Gaussian (training)`
`Test Image -> CNN Features -> Mahalanobis Distance vs Gaussian -> Anomaly Score Map -> Heatmap + Mask`

**Before running:** put your dataset on Google Drive in this layout (MVTec-AD style):

```
/content/drive/MyDrive/fabric_dataset/
    train/
        good/                 <- ONLY normal (defect-free) fabric images
    test/
        good/                 <- normal test images
        defect/               <- any name(s), folders of images with real defects
    ground_truth/             <- OPTIONAL pixel-level masks
        defect/                <- masks matching filenames in test/defect
```

If you don't have `ground_truth/` masks, that's fine — the notebook still works, you just won't get pixel-level AUC.

Run the cells top to bottom. Training only needs to be run once; after that you can reload the saved
Gaussian parameters and jump straight to the "Testing" section.

## 1. Mount Google Drive

In [ ]:
# Mount Google Drive so we can read the fabric dataset and write the trained model / results back to it
from google.colab import drive
drive.mount('/content/drive')

## 2. Install / confirm dependencies

In [ ]:
# Colab already ships with torch, torchvision, numpy, scipy, opencv, matplotlib, sklearn.
# This just makes sure everything we need is present (tqdm for progress bars in particular).
!pip install -q torch torchvision scikit-learn scipy opencv-python matplotlib tqdm

## 3. Imports and global setup

In [ ]:
import os                       # filesystem path handling
import random                   # random seeds + random dimension sampling (PaDiM's dim-reduction trick)
import pickle                   # save / load the trained Gaussian parameters
import numpy as np              # array math, linear algebra
import cv2                      # image loading/resizing, colormap for heatmaps
import torch                    # deep learning framework
import torch.nn.functional as F # functional ops: interpolate (resize) feature maps
from torch.utils.data import Dataset, DataLoader        # custom dataset + batching
from torchvision import transforms                      # image preprocessing pipeline
from torchvision.models import resnet18, wide_resnet50_2 # pretrained CNN backbones
from scipy.ndimage import gaussian_filter                # smooth the final anomaly heatmap
from sklearn.metrics import roc_auc_score                 # optional evaluation metric
import matplotlib.pyplot as plt                            # plotting heatmaps / results
from tqdm import tqdm                                        # progress bars

# fix random seeds everywhere so results are reproducible run to run
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# use the GPU if Colab gave us one (Runtime -> Change runtime type -> GPU / T4)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

## 4. Configuration — edit these for your dataset

In [ ]:
# ---------------- EDIT THESE PATHS / SETTINGS FOR YOUR OWN DATASET ----------------
DATASET_ROOT = '/content/drive/MyDrive/fabric_dataset'   # <-- change to your dataset folder
TRAIN_DIR    = os.path.join(DATASET_ROOT, 'train', 'good')   # normal-only training images
TEST_DIR     = os.path.join(DATASET_ROOT, 'test')             # subfolders: good/, defect_type_1/, ...
GT_DIR       = os.path.join(DATASET_ROOT, 'ground_truth')     # set to None if you have no pixel masks

# where to save the learned Gaussian parameters and test results
SAVE_DIR = os.path.join(DATASET_ROOT, 'padim_outputs')
os.makedirs(SAVE_DIR, exist_ok=True)

# backbone: 'resnet18' (fast, lighter, good default) or 'wide_resnet50_2' (paper's best accuracy, slower)
BACKBONE = 'resnet18'

# image size fed to the CNN (resize shorter side, then center-crop to a square)
RESIZE_SIZE = 256
CROP_SIZE   = 224

# number of randomly-selected feature channels to keep after concatenating layer1+layer2+layer3.
# This is PaDiM's dimensionality-reduction trick (random selection works ~as well as PCA and is
# much cheaper). Paper defaults: 100 for ResNet18, 550 for WideResNet50.
N_FEATURES = 100 if BACKBONE == 'resnet18' else 550

BATCH_SIZE = 32

## 5. Dataset class

In [ ]:
# Standard ImageNet normalization, required because we're using an ImageNet-pretrained backbone
IMG_TRANSFORM = transforms.Compose([
    transforms.ToPILImage(),                          # cv2 gives a numpy array -> convert to PIL for torchvision
    transforms.Resize(RESIZE_SIZE),                    # resize shorter side to RESIZE_SIZE
    transforms.CenterCrop(CROP_SIZE),                   # crop to a fixed square input size
    transforms.ToTensor(),                                # convert to tensor, scales pixels into [0, 1]
    transforms.Normalize(mean=[0.485, 0.456, 0.406],      # ImageNet per-channel means
                          std=[0.229, 0.224, 0.225]),       # ImageNet per-channel stds
])

MASK_TRANSFORM = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(RESIZE_SIZE),
    transforms.CenterCrop(CROP_SIZE),
    transforms.ToTensor(),   # keep mask values in [0, 1], no normalization needed for a mask
])


class FabricDataset(Dataset):
    """
    is_train=True  -> loads every image in `img_dir` as a normal ('good') sample.
    is_train=False -> `img_dir` has one subfolder per class (e.g. good/, hole/, stain/);
                       returns (image, label, mask, path) where label is 0=good, 1=defect,
                       and mask is the ground-truth defect mask if available, else all zeros.
    """
    def __init__(self, img_dir, is_train=True, gt_dir=None):
        self.is_train = is_train
        self.gt_dir = gt_dir
        self.samples = []  # list of (image_path, label, mask_path_or_None)

        if is_train:
            for fname in sorted(os.listdir(img_dir)):
                self.samples.append((os.path.join(img_dir, fname), 0, None))
        else:
            for cls in sorted(os.listdir(img_dir)):
                cls_dir = os.path.join(img_dir, cls)
                if not os.path.isdir(cls_dir):
                    continue
                label = 0 if cls.lower() == 'good' else 1   # anything not named 'good' counts as defective
                for fname in sorted(os.listdir(cls_dir)):
                    img_path = os.path.join(cls_dir, fname)
                    mask_path = None
                    if label == 1 and gt_dir is not None:
                        candidate = os.path.join(gt_dir, cls, fname)
                        if os.path.exists(candidate):
                            mask_path = candidate
                    self.samples.append((img_path, label, mask_path))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label, mask_path = self.samples[idx]

        img = cv2.imread(img_path)                       # OpenCV reads as BGR
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)         # convert to RGB, what torchvision expects
        img_tensor = IMG_TRANSFORM(img)

        if mask_path is not None:
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            mask_tensor = MASK_TRANSFORM(mask)
            mask_tensor = (mask_tensor > 0.5).float()      # binarize (in case mask has anti-aliased edges)
        else:
            mask_tensor = torch.zeros((1, CROP_SIZE, CROP_SIZE))  # no mask available -> all-zero placeholder

        return img_tensor, label, mask_tensor, img_path

## 6. Build the dataloaders

In [ ]:
train_dataset = FabricDataset(TRAIN_DIR, is_train=True)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

test_dataset = FabricDataset(TEST_DIR, is_train=False, gt_dir=GT_DIR)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Train (normal-only) images: {len(train_dataset)}')
print(f'Test images: {len(test_dataset)}')

## 7. Feature extractor — pretrained CNN + forward hooks

This is the *Pretrained CNN -> Extract Multi-layer Features* step.

In [ ]:
class FeatureExtractor:
    """
    Wraps a pretrained CNN and captures the intermediate feature maps from
    layer1, layer2, layer3 via forward hooks. These 3 maps are exactly the
    'Multi-layer Features' block in the PaDiM pipeline.
    """
    def __init__(self, backbone='resnet18'):
        if backbone == 'resnet18':
            self.model = resnet18(weights='IMAGENET1K_V1')
        else:
            self.model = wide_resnet50_2(weights='IMAGENET1K_V1')

        self.model.to(DEVICE)
        self.model.eval()                     # inference mode: freezes batchnorm/dropout behaviour
        for p in self.model.parameters():
            p.requires_grad = False            # backbone is never trained, only used for feature extraction

        self.features = []                      # will hold [layer1_out, layer2_out, layer3_out] per forward pass

        # every time layer1 / layer2 / layer3 finishes computing during a forward pass,
        # `_hook` is called automatically and stores its output
        self.model.layer1.register_forward_hook(self._hook)
        self.model.layer2.register_forward_hook(self._hook)
        self.model.layer3.register_forward_hook(self._hook)

    def _hook(self, module, input, output):
        self.features.append(output)

    def __call__(self, x):
        self.features = []                      # clear features left over from the previous batch
        with torch.no_grad():                    # no gradients needed -> saves memory and time
            _ = self.model(x.to(DEVICE))          # forward pass; hooks fire and populate self.features
        return self.features                      # [feat_layer1, feat_layer2, feat_layer3]

## 8. Feature embedding — fuse the 3 layers into one tensor per spatial location

In [ ]:
def embedding_concat(features):
    """
    Combines the 3 multi-scale feature maps into one embedding vector per
    spatial location -- the 'Feature Embedding' step of PaDiM.
    layer2 and layer3 are smaller than layer1, so they're upsampled with
    nearest-neighbor interpolation to layer1's resolution, then all three
    are concatenated along the channel dimension.
    """
    layer1, layer2, layer3 = features
    target_h, target_w = layer1.shape[2], layer1.shape[3]   # layer1 has the largest spatial resolution

    layer2_up = F.interpolate(layer2, size=(target_h, target_w), mode='nearest')
    layer3_up = F.interpolate(layer3, size=(target_h, target_w), mode='nearest')

    embedding = torch.cat([layer1, layer2_up, layer3_up], dim=1)  # (B, C1+C2+C3, H, W)
    return embedding


def extract_embeddings(dataloader, extractor, desc='Extracting'):
    """Runs every image in dataloader through the backbone, returns (N_images, C, H, W)."""
    all_embeddings = []
    for imgs, *_ in tqdm(dataloader, desc=desc):
        feats = extractor(imgs)                 # get layer1/2/3 feature maps for this batch
        emb = embedding_concat(feats)             # fuse them into one embedding tensor
        all_embeddings.append(emb.cpu())          # move to CPU so GPU memory doesn't fill up
    return torch.cat(all_embeddings, dim=0)        # stack every batch -> (N, C, H, W)

## 9. Training — estimate a Gaussian distribution for every patch position

This is `Estimate Gaussian Distribution for every spatial location`, and is where **training ends**.
For every one of the H×W grid positions in the feature map, we fit `N(mean, covariance)` across all
normal training images. At test time an unusual (defective) patch will land far from this learned
distribution.

In [ ]:
extractor = FeatureExtractor(BACKBONE)

# 1) run every training (normal-only) image through the CNN and fuse the multi-layer features
train_embeddings = extract_embeddings(train_loader, extractor, desc='Extracting train features')
N, C, H, W = train_embeddings.shape
print('Embedding tensor shape (N, C, H, W):', train_embeddings.shape)

# 2) randomly keep N_FEATURES of the C channels (PaDiM's cheap alternative to PCA)
random.seed(SEED)
selected_idx = torch.tensor(random.sample(range(C), N_FEATURES))
train_embeddings = torch.index_select(train_embeddings, dim=1, index=selected_idx)

# reshape so every (h, w) grid position becomes one 'patch' with an (N, N_FEATURES) set of vectors
train_embeddings = train_embeddings.view(N, N_FEATURES, H * W)

# 3) fit N(mean, covariance) independently for every patch position
mean = torch.mean(train_embeddings, dim=0).numpy()           # shape (N_FEATURES, H*W)
cov  = np.zeros((N_FEATURES, N_FEATURES, H * W))                # one covariance matrix per patch position
identity = np.identity(N_FEATURES)

for i in tqdm(range(H * W), desc='Estimating Gaussian per patch'):
    patch_vectors = train_embeddings[:, :, i].numpy()           # (N, N_FEATURES): this patch across all training images
    # sample covariance + small regularization (0.01*I) so it's always invertible even with limited data
    cov[:, :, i] = np.cov(patch_vectors, rowvar=False) + 0.01 * identity

# save everything needed at test time
with open(os.path.join(SAVE_DIR, 'padim_params.pkl'), 'wb') as f:
    pickle.dump({'mean': mean, 'cov': cov, 'selected_idx': selected_idx,
                 'H': H, 'W': W, 'backbone': BACKBONE, 'n_features': N_FEATURES}, f)

print('Training complete. Saved Gaussian parameters to', os.path.join(SAVE_DIR, 'padim_params.pkl'))

## 10. Testing — Mahalanobis distance vs the learned Gaussian

You can start here directly in a fresh session (after step 1-4 and defining `FeatureExtractor` /
`embedding_concat` / `extract_embeddings` above) as long as `padim_params.pkl` already exists on Drive —
no need to retrain every time.

In [ ]:
# reload the trained parameters (works even in a brand-new Colab session)
with open(os.path.join(SAVE_DIR, 'padim_params.pkl'), 'rb') as f:
    params = pickle.load(f)
mean          = params['mean']
cov           = params['cov']
selected_idx  = params['selected_idx']
H, W          = params['H'], params['W']
N_FEATURES    = params['n_features']

if 'extractor' not in globals():
    extractor = FeatureExtractor(params['backbone'])

# pre-compute the inverse of every patch's covariance matrix once (Mahalanobis distance needs Sigma^-1)
print('Inverting covariance matrices...')
cov_inv = np.zeros_like(cov)
for i in tqdm(range(H * W)):
    cov_inv[:, :, i] = np.linalg.inv(cov[:, :, i])


def compute_anomaly_maps(dataloader, extractor):
    """
    For every test image, extracts its embedding, then for every patch position computes the
    Mahalanobis distance to that patch's learned Gaussian N(mean, cov). A high distance means
    that patch looks statistically unlike anything seen in the normal training data.
    Returns score_maps with shape (N_test_images, H, W).
    """
    embeddings = extract_embeddings(dataloader, extractor, desc='Extracting test features')
    embeddings = torch.index_select(embeddings, dim=1, index=selected_idx)   # keep the same channels as training
    N_test = embeddings.shape[0]
    embeddings = embeddings.view(N_test, N_FEATURES, H * W).numpy()

    diff = embeddings - mean[np.newaxis, :, :]     # (N_test, C, HW): distance of each test patch from its learned mean
    # squared Mahalanobis distance for every image & every patch, fully vectorized (no Python-level double loop):
    # for each image n and patch p: diff[n,:,p]^T @ cov_inv[:,:,p] @ diff[n,:,p]
    dist_sq = np.einsum('ncp,cdp,ndp->np', diff, cov_inv, diff)
    dist = np.sqrt(np.clip(dist_sq, a_min=0, a_max=None))  # clip guards tiny negative values from floating-point error
    return dist.reshape(N_test, H, W)


# gather labels / ground-truth masks / file paths alongside the score maps
all_labels, all_masks, all_paths = [], [], []
for _, labels, masks, paths in test_loader:
    all_labels.extend(labels.tolist())
    all_masks.append(masks)
    all_paths.extend(paths)
all_masks = torch.cat(all_masks, dim=0).numpy()

score_maps = compute_anomaly_maps(test_loader, extractor)
print('Raw anomaly score maps shape:', score_maps.shape)

## 11. Anomaly Score Map — upsample to full image size and smooth

In [ ]:
def postprocess_score_maps(score_maps, out_size=CROP_SIZE, sigma=4):
    """
    score_maps: (N, H, W) low-resolution distance maps straight from the CNN's feature grid.
    Upsamples each one to (out_size, out_size) with bilinear interpolation (so it lines up with
    the original image pixels), then Gaussian-blurs it to remove blocky artifacts -- this produces
    the final anomaly heatmap.
    """
    maps = torch.tensor(score_maps).unsqueeze(1)                                  # (N, 1, H, W): add channel dim for interpolate
    maps = F.interpolate(maps, size=out_size, mode='bilinear', align_corners=False)
    maps = maps.squeeze(1).numpy()                                                  # back to (N, out_size, out_size)

    smoothed = np.zeros_like(maps)
    for i in range(maps.shape[0]):
        smoothed[i] = gaussian_filter(maps[i], sigma=sigma)   # smooth pixel-level noise into a clean heatmap
    return smoothed

anomaly_maps = postprocess_score_maps(score_maps)
print('Final anomaly maps shape:', anomaly_maps.shape)

# one overall anomaly score per image = the peak value in its smoothed heatmap (standard PaDiM choice)
image_scores = anomaly_maps.reshape(anomaly_maps.shape[0], -1).max(axis=1)

def normalize(x):
    # min-max scale to [0, 1], purely for nicer visualization -- doesn't change ranking or AUC
    return (x - x.min()) / (x.max() - x.min() + 1e-8)

norm_maps = np.array([normalize(m) for m in anomaly_maps])
norm_image_scores = normalize(image_scores)

## 12. (Optional) Evaluation — ROC-AUC, if your test set has both good and defective images

In [ ]:
all_labels_np = np.array(all_labels)

if len(set(all_labels_np.tolist())) > 1:
    image_auc = roc_auc_score(all_labels_np, image_scores)
    print(f'Image-level ROC-AUC: {image_auc:.4f}')

    if GT_DIR is not None and all_masks.sum() > 0:
        pixel_auc = roc_auc_score(all_masks.flatten(), anomaly_maps.flatten())
        print(f'Pixel-level ROC-AUC: {pixel_auc:.4f}')
    else:
        print('No ground-truth masks found -- skipping pixel-level AUC.')
else:
    print('Only one class present in the test set -- need both good & defective samples for ROC-AUC.')

## 13. Heatmap + Segmentation Mask — final visualization

In [ ]:
def get_binary_mask(norm_map, threshold=0.5):
    """Thresholds the normalized heatmap to produce a binary defect segmentation mask."""
    return (norm_map > threshold).astype(np.uint8) * 255


def visualize_result(idx, threshold=0.5):
    img_path = all_paths[idx]
    orig = cv2.imread(img_path)
    orig = cv2.cvtColor(orig, cv2.COLOR_BGR2RGB)
    orig = cv2.resize(orig, (CROP_SIZE, CROP_SIZE))

    heatmap = cv2.applyColorMap((norm_maps[idx] * 255).astype(np.uint8), cv2.COLORMAP_JET)  # colorize the score map
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(orig, 0.6, heatmap, 0.4, 0)   # blend heatmap over the original image

    mask = get_binary_mask(norm_maps[idx], threshold)

    fig, axs = plt.subplots(1, 4, figsize=(16, 4))
    axs[0].imshow(orig);                          axs[0].set_title('Original');           axs[0].axis('off')
    axs[1].imshow(norm_maps[idx], cmap='jet');     axs[1].set_title('Anomaly Score Map');   axs[1].axis('off')
    axs[2].imshow(overlay);                        axs[2].set_title('Heatmap Overlay');      axs[2].axis('off')
    axs[3].imshow(mask, cmap='gray');               axs[3].set_title('Segmentation Mask');    axs[3].axis('off')
    label_str = 'DEFECT' if all_labels[idx] == 1 else 'GOOD'
    plt.suptitle(f'Image score: {norm_image_scores[idx]:.3f}  |  Label: {label_str}')
    plt.tight_layout()
    plt.show()

# visualize a handful of random test images
sample_idxs = random.sample(range(len(all_paths)), min(5, len(all_paths)))
for i in sample_idxs:
    visualize_result(i, threshold=0.5)   # tune `threshold` (0-1) to make the mask stricter/looser

## 14. Save results back to Drive

In [ ]:
# save the numeric results so they survive a Colab runtime reset
np.savez(os.path.join(SAVE_DIR, 'test_results.npz'),
         anomaly_maps=anomaly_maps,
         image_scores=image_scores,
         labels=np.array(all_labels),
         paths=np.array(all_paths))
print('Saved results to', os.path.join(SAVE_DIR, 'test_results.npz'))

## Notes / tuning tips

- **`threshold`** in `visualize_result` controls how much of the heatmap counts as "defect" in the binary
  mask — lower it if real defects are being missed, raise it if too much normal fabric is flagged.
- **`BACKBONE = 'wide_resnet50_2'`** gives noticeably better accuracy than ResNet18 at the cost of more
  GPU memory / slower extraction — worth trying once the ResNet18 pipeline is working end-to-end.
- **No ground-truth masks?** That's fine — everything works, you just won't get a pixel-level ROC-AUC
  number; the heatmap/mask visualization still works from the learned Gaussian alone.
- **Very large training sets**: the per-patch covariance loop (`H*W` iterations) is the slowest part of
  training; it typically takes well under a minute even for a few hundred images at 224x224.